In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

from pathlib import Path
from collections import Counter

print("Libraries imported successfully!")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
BASE_DIR = Path(
    r"C:\Users\Admin\ML\anamoly\mvtec_anomaly_detection"
)

CATEGORY = "toothbrush"
DATA_DIR = BASE_DIR / CATEGORY

TRAIN_DIR = DATA_DIR / "train" / "good"
TEST_DIR = DATA_DIR / "test"
GROUND_TRUTH_DIR = DATA_DIR / "ground_truth"

IMAGE_SIZE = 128
BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 0.001

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

OUTPUT_DIR = BASE_DIR / "outputs" / CATEGORY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Category:", CATEGORY)
print("Training folder:", TRAIN_DIR)
print("Testing folder:", TEST_DIR)
print("Device:", DEVICE)
print("Output folder:", OUTPUT_DIR)

In [ ]:
from pathlib import Path
import os

print("Current working directory:")
print(os.getcwd())

print("\nFolders in the current directory:")
for item in Path.cwd().iterdir():
    print(item.name)

In [ ]:
from pathlib import Path
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

BASE_DIR = Path(
    r"C:\Users\Admin\ML\analomy\mvtec_anomaly_detection"
)

CATEGORY = "toothbrush"

DATA_DIR = BASE_DIR / CATEGORY
TRAIN_DIR = DATA_DIR / "train" / "good"
TEST_DIR = DATA_DIR / "test"
GROUND_TRUTH_DIR = DATA_DIR / "ground_truth"

IMAGE_SIZE = 128
BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 0.001

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

OUTPUT_DIR = BASE_DIR / "outputs" / CATEGORY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset exists:", DATA_DIR.exists())
print("Training folder exists:", TRAIN_DIR.exists())
print("Testing folder exists:", TEST_DIR.exists())
print("Ground-truth folder exists:", GROUND_TRUTH_DIR.exists())
print("Device:", DEVICE)
print("Output folder:", OUTPUT_DIR)

In [ ]:
from pathlib import Path

IMAGE_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"
}

def get_image_files(folder):
    folder = Path(folder)

    return sorted(
        path for path in folder.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )

print("Image loading function created successfully!")
print("Training images:", len(get_image_files(TRAIN_DIR)))
print("Test images:", len(get_image_files(TEST_DIR)))

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

train_files = get_image_files(TRAIN_DIR)
test_files = get_image_files(TEST_DIR)

good_test_files = get_image_files(TEST_DIR / "good")

defect_test_files = [
    path for path in test_files
    if path.parent.name != "good"
]

def show_samples(files, title, count=4):
    if len(files) == 0:
        print("No images found:", title)
        return

    selected = random.sample(files, min(count, len(files)))

    fig, axes = plt.subplots(1, len(selected), figsize=(12, 4))
    axes = np.atleast_1d(axes)

    for ax, path in zip(axes, selected):
        ax.imshow(Image.open(path).convert("RGB"))
        ax.set_title(path.parent.name)
        ax.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

show_samples(train_files, "Normal Training Images")
show_samples(good_test_files, "Normal Test Images")
show_samples(defect_test_files, "Defective Test Images")

In [ ]:
IMAGE_SIZE = 128

def load_image(path, size=IMAGE_SIZE):
    image = Image.open(path).convert("RGB")
    image = image.resize(
        (size, size),
        Image.Resampling.BILINEAR
    )

    array = np.asarray(image, dtype=np.float32) / 255.0

    tensor = torch.from_numpy(array).permute(2, 0, 1)

    return tensor

sample_tensor = load_image(train_files[0])

print("Image shape:", sample_tensor.shape)
print("Data type:", sample_tensor.dtype)
print("Minimum pixel value:", sample_tensor.min().item())
print("Maximum pixel value:", sample_tensor.max().item())

plt.figure(figsize=(5, 5))
plt.imshow(sample_tensor.permute(1, 2, 0).numpy())
plt.title("Preprocessed Toothbrush Image")
plt.axis("off")
plt.show()

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MVTecDataset(Dataset):
    def __init__(self, files):
        self.files = list(files)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        path = self.files[index]
        image = load_image(path)

        label = 0 if path.parent.name == "good" else 1

        return image, label, str(path)


train_dataset_all = MVTecDataset(train_files)
test_dataset = MVTecDataset(test_files)

print("Training dataset:", len(train_dataset_all))
print("Test dataset:", len(test_dataset))
print("Image shape:", train_dataset_all[0][0].shape)

In [ ]:
import random
import numpy as np
import torch

SEED = 42
BATCH_SIZE = 8

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

validation_size = max(1, int(0.10 * len(train_dataset_all)))
training_size = len(train_dataset_all) - validation_size

generator = torch.Generator().manual_seed(SEED)

train_dataset, val_dataset = random_split(
    train_dataset_all,
    [training_size, validation_size],
    generator=generator
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))
print("Training batches:", len(train_loader))

In [ ]:
images, labels, paths = next(iter(train_loader))

print("Batch shape:", images.shape)
print("Data type:", images.dtype)
print("Minimum pixel value:", images.min().item())
print("Maximum pixel value:", images.max().item())

plt.figure(figsize=(5, 5))
plt.imshow(images[0].permute(1, 2, 0).numpy())
plt.title("Training Image")
plt.axis("off")
plt.show()

In [ ]:
import torch
import torch.nn as nn

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", DEVICE)

In [ ]:
class ConvAutoencoder(nn.Module):

    def __init__(self):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, 4, stride=2, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 256, 4, stride=2, padding=1),
            nn.ReLU(inplace=True)
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        reconstructed = self.decoder(encoded)
        return reconstructed


model = ConvAutoencoder().to(DEVICE)

print(model)

parameters = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("\nTrainable parameters:", f"{parameters:,}")

In [ ]:
model.eval()

with torch.no_grad():
    sample_images = images[:1].to(DEVICE)
    reconstructed = model(sample_images)

print("Input shape:", sample_images.shape)
print("Output shape:", reconstructed.shape)
print("Output minimum:", reconstructed.min().item())
print("Output maximum:", reconstructed.max().item())

In [ ]:
original = sample_images[0].cpu().permute(1, 2, 0).numpy()

reconstructed_image = (
    reconstructed[0].cpu().permute(1, 2, 0).numpy()
)

plt.figure(figsize=(9, 4))

plt.subplot(1, 2, 1)
plt.imshow(original)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(reconstructed_image)
plt.title("Reconstruction Before Training")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch.optim as optim

criterion = nn.MSELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

print("Loss function:", criterion)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", 0.001)

In [ ]:
EPOCHS = 30

history = {
    "train_loss": [],
    "val_loss": []
}

best_val_loss = float("inf")

best_model_path = OUTPUT_DIR / "best_autoencoder.pth"

for epoch in range(EPOCHS):

    # Training phase
    model.train()
    train_total = 0.0

    for batch_images, _, _ in train_loader:
        batch_images = batch_images.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(batch_images)
        loss = criterion(outputs, batch_images)

        loss.backward()
        optimizer.step()

        train_total += loss.item() * batch_images.size(0)

    train_loss = train_total / len(train_dataset)

    # Validation phase
    model.eval()
    val_total = 0.0

    with torch.no_grad():
        for batch_images, _, _ in val_loader:
            batch_images = batch_images.to(DEVICE)

            outputs = model(batch_images)
            loss = criterion(outputs, batch_images)

            val_total += loss.item() * batch_images.size(0)

    val_loss = val_total / len(val_dataset)

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            best_model_path
        )

    print(
        f"Epoch [{epoch + 1:02d}/{EPOCHS}] "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}"
    )

print("\nTraining completed!")
print("Best validation loss:", best_val_loss)
print("Best model saved at:", best_model_path)

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    history["train_loss"],
    label="Training Loss"
)

plt.plot(
    history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("CNN Autoencoder Training Performance")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=DEVICE,
        weights_only=True
    )
)

model.eval()

print("Best trained model loaded successfully!")

In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.plot(
    range(1, len(history["train_loss"]) + 1),
    history["train_loss"],
    label="Training Loss"
)

plt.plot(
    range(1, len(history["val_loss"]) + 1),
    history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Autoencoder Training Performance")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:

import numpy as np
import torch

def get_anomaly_map(original, reconstructed):
    # Calculate pixel-wise reconstruction error
    return torch.abs(original - reconstructed).mean(dim=1)

def get_image_score(error_map):
    # Average the largest 1% of pixel errors
    flat_errors = error_map.flatten(start_dim=1)

    k = max(1, int(np.ceil(flat_errors.shape[1] * 0.01)))

    return torch.topk(
        flat_errors, k, dim=1
    ).values.mean(dim=1)

print("Anomaly map and image scoring functions created successfully!")


In [ ]:

val_scores = []

model.eval()

with torch.no_grad():
    for batch_images, _, _ in val_loader:
        batch_images = batch_images.to(DEVICE)

        reconstructed = model(batch_images)
        error_map = get_anomaly_map(
            batch_images, reconstructed
        )

        scores = get_image_score(error_map)

        val_scores.extend(
            scores.cpu().numpy().tolist()
        )

ANOMALY_THRESHOLD = float(np.percentile(val_scores, 99))

print("Validation images:", len(val_scores))
print("Anomaly threshold:", ANOMALY_THRESHOLD)


In [ ]:

import pandas as pd

results = []

model.eval()

with torch.no_grad():
    for batch_images, labels, paths in test_loader:
        batch_images = batch_images.to(DEVICE)

        reconstructed = model(batch_images)

        error_map = get_anomaly_map(
            batch_images, reconstructed
        )

        scores = get_image_score(error_map)

        for i, path in enumerate(paths):
            score = float(scores[i].cpu().item())
            actual_label = int(labels[i].item())

            prediction = int(score > ANOMALY_THRESHOLD)

            results.append({
                "image_path": path,
                "actual_label": actual_label,
                "anomaly_score": score,
                "predicted_label": prediction,
                "actual_class": (
                    "Defective" if actual_label == 1 else "Normal"
                ),
                "predicted_class": (
                    "Defective" if prediction == 1 else "Normal"
                )
            })

results_df = pd.DataFrame(results)

print("ANOMALY DETECTION RESULTS")
print("=" * 50)
print("Total test images:", len(results_df))
print("Predicted normal:", (results_df["predicted_label"] == 0).sum())
print("Predicted defective:", (results_df["predicted_label"] == 1).sum())

display(results_df.head(10))


In [ ]:

%pip install scikit-learn


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

y_true = results_df["actual_label"]
y_pred = results_df["predicted_label"]

print("CLASSIFICATION REPORT")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["Normal", "Defective"],
    zero_division=0
))

print("CONFUSION MATRIX")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

print("OVERALL ACCURACY: {:.2f}%".format(
    accuracy_score(y_true, y_pred) * 100
))


In [ ]:

from sklearn.model_selection import train_test_split

# Labels: 0 = normal, 1 = defective
test_labels = [
    0 if path.parent.name == "good" else 1
    for path in test_files
]

calibration_files, final_test_files = train_test_split(
    test_files,
    test_size=0.60,
    random_state=SEED,
    stratify=test_labels
)

print("Total test images:", len(test_files))
print("Calibration images:", len(calibration_files))
print("Final evaluation images:", len(final_test_files))

print("\nCalibration class counts:")
print({
    "Normal": sum(p.parent.name == "good" for p in calibration_files),
    "Defective": sum(p.parent.name != "good" for p in calibration_files)
})

print("\nFinal evaluation class counts:")
print({
    "Normal": sum(p.parent.name == "good" for p in final_test_files),
    "Defective": sum(p.parent.name != "good" for p in final_test_files)
})


In [ ]:

def calculate_scores(file_list):
    scores = []
    labels = []

    model.eval()

    with torch.no_grad():
        for path in file_list:
            image = load_image(path).unsqueeze(0).to(DEVICE)
            reconstruction = model(image)

            error_map = get_anomaly_map(image, reconstruction)
            score = get_image_score(error_map).item()

            label = 0 if path.parent.name == "good" else 1

            scores.append(score)
            labels.append(label)

    return np.array(scores), np.array(labels)


calibration_scores, calibration_labels = calculate_scores(
    calibration_files
)

final_scores, final_labels = calculate_scores(final_test_files)

print("Calibration scores calculated:", len(calibration_scores))
print("Final evaluation scores calculated:", len(final_scores))


In [ ]:

from sklearn.metrics import accuracy_score

candidate_thresholds = np.unique(calibration_scores)

best_threshold = None
best_calibration_accuracy = -1

for threshold in candidate_thresholds:
    calibration_predictions = (
        calibration_scores > threshold
    ).astype(int)

    accuracy = accuracy_score(
        calibration_labels,
        calibration_predictions
    )

    if accuracy > best_calibration_accuracy:
        best_calibration_accuracy = accuracy
        best_threshold = float(threshold)

print("Original threshold:", ANOMALY_THRESHOLD)
print("New calibrated threshold:", best_threshold)
print("Calibration accuracy: {:.2f}%".format(
    best_calibration_accuracy * 100
))


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

final_predictions = (
    final_scores > best_threshold
).astype(int)

print("FINAL HELD-OUT EVALUATION")
print("=" * 40)

print("Accuracy: {:.2f}%".format(
    accuracy_score(final_labels, final_predictions) * 100
))

print("\nClassification report:")
print(classification_report(
    final_labels,
    final_predictions,
    labels=[0, 1],
    target_names=["Normal", "Defective"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(
    final_labels,
    final_predictions,
    labels=[0, 1]
))


In [ ]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

threshold_rows = []

for threshold in np.unique(calibration_scores):
    predictions = (calibration_scores > threshold).astype(int)

    normal_recall = recall_score(
        calibration_labels, predictions,
        pos_label=0, zero_division=0
    )

    defective_precision = precision_score(
        calibration_labels, predictions,
        pos_label=1, zero_division=0
    )

    defective_recall = recall_score(
        calibration_labels, predictions,
        pos_label=1, zero_division=0
    )

    accuracy = accuracy_score(
        calibration_labels, predictions
    )

    threshold_rows.append({
        "threshold": float(threshold),
        "accuracy": accuracy,
        "normal_recall": normal_recall,
        "defective_precision": defective_precision,
        "defective_recall": defective_recall
    })

threshold_df = pd.DataFrame(threshold_rows)

# First prefer thresholds satisfying the requested targets
eligible = threshold_df[
    (threshold_df["normal_recall"] >= 0.80) &
    (threshold_df["defective_precision"] >= 0.90)
]

if len(eligible) > 0:
    best_row = eligible.sort_values(
        ["accuracy", "defective_recall"],
        ascending=False
    ).iloc[0]

    print("Found a threshold meeting both calibration targets.")
else:
    # If impossible on this calibration subset, find the best trade-off
    threshold_df["target_score"] = (
        threshold_df["normal_recall"] +
        threshold_df["defective_precision"] +
        threshold_df["defective_recall"]
    ) / 3

    best_row = threshold_df.sort_values(
        ["target_score", "accuracy"],
        ascending=False
    ).iloc[0]

    print("No threshold meets both targets on calibration data.")
    print("Showing the best available trade-off instead.")

print("\nSelected threshold:", best_row["threshold"])
print("Calibration accuracy: {:.2f}%".format(
    best_row["accuracy"] * 100
))
print("Normal recall: {:.2f}%".format(
    best_row["normal_recall"] * 100
))
print("Defective precision: {:.2f}%".format(
    best_row["defective_precision"] * 100
))
print("Defective recall: {:.2f}%".format(
    best_row["defective_recall"] * 100
))


In [ ]:

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

selected_threshold = float(best_row["threshold"])

final_predictions_tuned = (
    final_scores > selected_threshold
).astype(int)

print("HELD-OUT RESULTS")
print("=" * 40)

print(classification_report(
    final_labels,
    final_predictions_tuned,
    labels=[0, 1],
    target_names=["Normal", "Defective"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(
    final_labels,
    final_predictions_tuned,
    labels=[0, 1]
))

print("Accuracy: {:.2f}%".format(
    accuracy_score(
        final_labels, final_predictions_tuned
    ) * 100
))


In [ ]:

def get_anomaly_map(original, reconstructed):
    return torch.abs(original - reconstructed).mean(dim=1)

def get_image_score(error_map):
    flat_errors = error_map.flatten(start_dim=1)

    # Average the largest 10% of pixel errors
    k = max(1, int(np.ceil(flat_errors.shape[1] * 0.10)))

    return torch.topk(
        flat_errors, k, dim=1
    ).values.mean(dim=1)

print("Updated anomaly scoring method: top 10% pixel errors")


In [ ]:

def calculate_scores(file_list):
    scores = []
    labels = []

    model.eval()

    with torch.no_grad():
        for path in file_list:
            image = load_image(path).unsqueeze(0).to(DEVICE)

            reconstructed = model(image)

            error_map = get_anomaly_map(image, reconstructed)

            score = get_image_score(error_map).item()

            label = 0 if path.parent.name == "good" else 1

            scores.append(score)
            labels.append(label)

    return np.array(scores), np.array(labels)

print("calculate_scores() defined successfully!")


In [ ]:

from pathlib import Path
import numpy as np
import torch
from sklearn.model_selection import train_test_split

# Dataset paths
BASE_DIR = Path(
    r"C:\Users\Admin\ML\analomy\mvtec_anomaly_detection"
)

CATEGORY = "toothbrush"
DATA_DIR = BASE_DIR / CATEGORY
TRAIN_DIR = DATA_DIR / "train" / "good"
TEST_DIR = DATA_DIR / "test"

IMAGE_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"
}

def get_image_files(folder):
    folder = Path(folder)
    return sorted(
        path for path in folder.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )

# Rebuild file lists
train_files = get_image_files(TRAIN_DIR)
test_files = get_image_files(TEST_DIR)

print("Training images:", len(train_files))
print("Test images:", len(test_files))

# Recreate calibration and evaluation split
SEED = 42

test_labels = np.array([
    0 if path.parent.name == "good" else 1
    for path in test_files
])

calibration_files, final_test_files = train_test_split(
    test_files,
    test_size=0.60,
    random_state=SEED,
    stratify=test_labels
)

print("Calibration images:", len(calibration_files))
print("Final evaluation images:", len(final_test_files))


In [ ]:

from pathlib import Path
from PIL import Image
import numpy as np
import torch
import torch.nn as nn

# Settings
IMAGE_SIZE = 128
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

OUTPUT_DIR = BASE_DIR / "outputs" / CATEGORY
MODEL_PATH = OUTPUT_DIR / "best_autoencoder.pth"

# Image preprocessing
def load_image(path, size=IMAGE_SIZE):
    image = Image.open(path).convert("RGB")
    image = image.resize(
        (size, size), Image.Resampling.BILINEAR
    )
    array = np.asarray(image, dtype=np.float32) / 255.0
    return torch.from_numpy(array).permute(2, 0, 1)


# Recreate the same autoencoder architecture
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1), nn.ReLU(inplace=True)
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 3, 4, 2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


model = ConvAutoencoder().to(DEVICE)

model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=DEVICE,
        weights_only=True
    )
)

model.eval()

print("Device:", DEVICE)
print("Saved autoencoder loaded successfully!")


In [ ]:
calibration_scores, calibration_labels = calculate_scores(
    calibration_files
)

final_scores, final_labels = calculate_scores(
    final_test_files
)

print("Calibration scores:", len(calibration_scores))
print("Evaluation scores:", len(final_scores))

In [ ]:

import pandas as pd
from sklearn.metrics import confusion_matrix, accuracy_score

threshold_rows = []

for threshold in np.unique(calibration_scores):
    calibration_predictions = (
        calibration_scores > threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        calibration_labels,
        calibration_predictions,
        labels=[0, 1]
    ).ravel()

    normal_recall = tn / (tn + fp) if tn + fp else 0
    defective_recall = tp / (tp + fn) if tp + fn else 0
    defective_precision = tp / (tp + fp) if tp + fp else 0
    accuracy = accuracy_score(
        calibration_labels, calibration_predictions
    )

    threshold_rows.append({
        "threshold": float(threshold),
        "accuracy": accuracy,
        "normal_recall": normal_recall,
        "defective_recall": defective_recall,
        "defective_precision": defective_precision
    })

threshold_df = pd.DataFrame(threshold_rows)

# Check whether any threshold meets all target metrics
eligible = threshold_df[
    (threshold_df["normal_recall"] >= 0.80) &
    (threshold_df["defective_recall"] >= 0.90) &
    (threshold_df["defective_precision"] >= 0.90)
]

if not eligible.empty:
    selected = eligible.sort_values(
        "accuracy", ascending=False
    ).iloc[0]
    print("All calibration targets met.")
else:
    # Otherwise, prioritize a balance between normal and defect recall
    threshold_df["balanced_recall"] = (
        threshold_df["normal_recall"] +
        threshold_df["defective_recall"]
    ) / 2

    selected = threshold_df.sort_values(
        ["balanced_recall", "defective_precision", "accuracy"],
        ascending=False
    ).iloc[0]

    print("No threshold meets every target on calibration data.")

best_threshold = float(selected["threshold"])

print("\nSelected threshold:", best_threshold)
display(selected.to_frame("Selected metrics"))


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Use the threshold selected from the calibration set
final_predictions = (final_scores > best_threshold).astype(int)

print("=" * 55)
print("FINAL TEST SET PERFORMANCE")
print("=" * 55)

print(f"Threshold: {best_threshold:.6f}")
print(f"Test images: {len(final_labels)}")
print(f"Accuracy: {accuracy_score(final_labels, final_predictions) * 100:.2f}%")

print("\nClassification Report:")
print(
    classification_report(
        final_labels,
        final_predictions,
        labels=[0, 1],
        target_names=["Normal", "Defective"],
        zero_division=0
    )
)

print("Confusion Matrix:")
print(confusion_matrix(
    final_labels,
    final_predictions,
    labels=[0, 1]
))

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image

# Project paths
PROJECT_DIR = Path(r"C:\Users\Admin\ML\analomy")
BASE_DIR = PROJECT_DIR / "mvtec_anomaly_detection"
CATEGORY = "toothbrush"

DATA_DIR = BASE_DIR / CATEGORY
TEST_DIR = DATA_DIR / "test"
OUTPUT_DIR = PROJECT_DIR / "outputs" / CATEGORY
VIS_DIR = OUTPUT_DIR / "anomaly_visualizations"

VIS_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset exists:", DATA_DIR.exists())
print("Test folder exists:", TEST_DIR.exists())
print("Visualization folder:", VIS_DIR)
print("Device:", DEVICE if "DEVICE" in globals() else "Not initialized yet")

In [ ]:
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, 4, 2, 1),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, 4, 2, 1),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 256, 4, 2, 1),
            nn.ReLU(inplace=True)
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(32, 3, 4, 2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


checkpoint_path = OUTPUT_DIR / "best_autoencoder.pth"

if "model" not in globals():
    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Model checkpoint not found: {checkpoint_path}\n"
            "Check where best_autoencoder.pth was saved."
        )

    model = ConvAutoencoder().to(DEVICE)
    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=True
    )

    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    elif "state_dict" in checkpoint:
        model.load_state_dict(checkpoint["state_dict"])
    else:
        model.load_state_dict(checkpoint)

    print("Trained model loaded successfully.")
else:
    model = model.to(DEVICE)
    print("Using existing trained model.")

model.eval()
print("Device:", DEVICE)

In [ ]:
def load_image(path, size=128):
    image = Image.open(path).convert("RGB")
    image = image.resize((size, size), Image.Resampling.BILINEAR)

    array = np.asarray(image, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1)

    return tensor


def get_anomaly_map(original, reconstructed):
    # Average absolute error across RGB channels
    return torch.abs(original - reconstructed).mean(dim=1)


def get_image_score(error_map):
    flat_errors = error_map.flatten(start_dim=1)
    k = max(1, int(np.ceil(flat_errors.shape[1] * 0.10)))

    return torch.topk(flat_errors, k, dim=1).values.mean(dim=1)


THRESHOLD = 0.09615842998027802

print("Image size: 128 × 128")
print("Anomaly threshold:", THRESHOLD)
print("Image processing functions are ready.")

In [ ]:
def get_image_files(folder):
    extensions = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

    return sorted(
        p for p in Path(folder).rglob("*")
        if p.is_file() and p.suffix.lower() in extensions
    )


normal_files = get_image_files(TEST_DIR / "good")

defective_files = [
    p for p in get_image_files(TEST_DIR)
    if p.parent.name != "good"
]

print("Normal test images:", len(normal_files))
print("Defective test images:", len(defective_files))

if not normal_files:
    print("Warning: no images found in test/good.")

if not defective_files:
    print("Warning: no defective images found under test/.")

In [ ]:
def visualize_anomaly(path, save=True):
    image_tensor = load_image(path).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        reconstructed_tensor = model(image_tensor)
        error_tensor = get_anomaly_map(
            image_tensor, reconstructed_tensor
        )
        score = get_image_score(error_tensor).item()

    original = image_tensor[0].cpu().permute(1, 2, 0).numpy()
    reconstructed = (
        reconstructed_tensor[0].cpu().permute(1, 2, 0).numpy()
    )
    error_map = error_tensor[0].cpu().numpy()

    # Normalize heatmap for visualization only.
    # The classification score remains unchanged.
    heatmap = error_map / (error_map.max() + 1e-8)

    overlay = 0.6 * original + 0.4 * plt.get_cmap("jet")(heatmap)[..., :3]
    overlay = np.clip(overlay, 0, 1)

    prediction = "Defective" if score > THRESHOLD else "Normal"
    actual = "Normal" if path.parent.name == "good" else "Defective"

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    axes[0].imshow(original)
    axes[0].set_title("Original")

    axes[1].imshow(reconstructed)
    axes[1].set_title("Reconstruction")

    im = axes[2].imshow(error_map, cmap="hot")
    axes[2].set_title("Anomaly heatmap")
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    axes[3].imshow(overlay)
    axes[3].set_title("Heatmap overlay")

    for ax in axes:
        ax.axis("off")

    fig.suptitle(
        f"{path.name} | Actual: {actual} | "
        f"Predicted: {prediction} | Score: {score:.5f}",
        fontsize=11
    )
    plt.tight_layout()

    if save:
        output_path = VIS_DIR / f"{path.stem}_visualization.png"
        fig.savefig(output_path, dpi=150, bbox_inches="tight")
        print("Saved:", output_path)

    plt.show()
    plt.close(fig)

    return {
        "filename": path.name,
        "actual": actual,
        "prediction": prediction,
        "score": score,
        "correct": actual == prediction
    }


# Display one example from each class
if normal_files:
    normal_result = visualize_anomaly(normal_files[0])

if defective_files:
    defective_result = visualize_anomaly(defective_files[0])

In [ ]:
selected_files = normal_files[:5] + defective_files[:5]

saved_results = []

for path in selected_files:
    result = visualize_anomaly(path, save=True)
    saved_results.append(result)

print("=" * 50)
print("VISUALIZATION EXPORT COMPLETE")
print("=" * 50)
print("Images processed:", len(saved_results))
print("Output folder:", VIS_DIR)

In [ ]:
import pandas as pd

results_df = pd.DataFrame(saved_results)

csv_path = VIS_DIR / "visualization_predictions.csv"
results_df.to_csv(csv_path, index=False)

display(results_df)

print("CSV saved to:", csv_path)

In [ ]:
all_test_files = normal_files + defective_files
all_results = []

for index, path in enumerate(all_test_files, start=1):
    result = visualize_anomaly(path, save=True)
    all_results.append(result)

    if index % 10 == 0 or index == len(all_test_files):
        print(f"Processed {index}/{len(all_test_files)} images")

all_results_df = pd.DataFrame(all_results)

all_csv_path = VIS_DIR / "all_test_predictions.csv"
all_results_df.to_csv(all_csv_path, index=False)

print("\nAll test images processed.")
print("Visualizations:", VIS_DIR)
print("Prediction report:", all_csv_path)